In [7]:
!pip install -q langgraph langchain-core langchain-openai pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.4/146.4 kB 5.8 MB/s eta 0:00:00


In [9]:
import os
from google.colab import userdata

# Or if using Gemini:
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

Step 3: The First Code Block (State + Intake Node)

In [10]:
from typing import Annotated, List, Optional, TypedDict
from pydantic import BaseModel, Field
import operator

# ==========================================
# 1. DEFINE STRUCTURED SCHEMAS & STATE
# ==========================================

class IntakeExtraction(BaseModel):
    client_name: Optional[str] = Field(default=None, description="Client name if stated")
    legal_domain: str = Field(description="Domain: Labor, Civil, Criminal, Family, or Commercial")
    case_summary: str = Field(description="Factual, concise summary of the legal issue")
    urgency_level: str = Field(
        description="Must be 'Critical' (court date, police, or deadline < 72h), 'Medium', or 'Low'"
    )
    search_keywords: str = Field(description="Keywords and legal articles for RAG lookup")

class LegalAgentState(TypedDict):
    # Appends new messages rather than overwriting
    messages: Annotated[List[dict], operator.add]
    sender_id: str
    intake_data: Optional[IntakeExtraction]
    retrieved_statutes: List[str]
    draft_response: str
    is_escalated: bool

In [8]:
# !pip install langchain_google_genai

In [11]:
# ==========================================
# 2. SETUP THE LLM WITH STRUCTURED OUTPUT
# ==========================================

from langchain_openai import ChatOpenAI
# Alternatively for Gemini:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)



# Bind the Pydantic schema to force guaranteed JSON extraction
structured_parser = llm.with_structured_output(IntakeExtraction)

In [14]:
# 3. DEFINE THE INTAKE NODE
# ==========================================

def intake_node(state: LegalAgentState) -> dict:
    """
    Parses the inbound message, categorizes the domain, and evaluates urgency.
    """
    last_message = state["messages"][-1]["content"]

    prompt = f"""
    You are an administrative legal intake officer.
    Analyze the following incoming client message and extract the required legal intake fields.

    Client Message:
    \"\"\"{last_message}\"\"\"
    """

    extracted: IntakeExtraction = structured_parser.invoke(prompt)

    # Return the dictionary update to merge into state
    return {
        "intake_data": extracted,
        "is_escalated": extracted.urgency_level == "Critical"
    }



In [15]:
# ==========================================
# 4. RUN A TEST RUN IN COLAB
# ==========================================

# Simulated raw WhatsApp message from a frantic client
sample_incoming_message = (
    "Hello, my name is Tarek. My landlord just locked my apartment door "
    "and threw my things outside without an eviction order! "
    "I have an urgent police summons tomorrow morning at 9 AM. What do I do?"
)

initial_state: LegalAgentState = {
    "messages": [{"role": "user", "content": sample_incoming_message}],
    "sender_id": "+201012345678",
    "intake_data": None,
    "retrieved_statutes": [],
    "draft_response": "",
    "is_escalated": False
}

# Run the node directly
result = intake_node(initial_state)

print("--- Extracted Intake Data ---")
print(f"Client Name:   {result['intake_data'].client_name}")
print(f"Legal Domain:  {result['intake_data'].legal_domain}")
print(f"Urgency Level: {result['intake_data'].urgency_level}")
print(f"Summary:       {result['intake_data'].case_summary}")
print(f"Search Query:  {result['intake_data'].search_keywords}")
print(f"Escalated?     {result['is_escalated']}")

--- Extracted Intake Data ---
Client Name:   Tarek
Legal Domain:  Civil
Urgency Level: Critical
Summary:       Client's landlord illegally locked his apartment door and removed his belongings without a valid eviction order.
Search Query:  illegal eviction, landlord tenant law, unlawful detainer, tenant rights, police summons, property law
Escalated?     True


In [16]:
from langgraph.graph import StateGraph, START, END

# ==========================================
# 1. DEFINE DOWNSTREAM NODES
# ==========================================

def escalate_urgent_node(state: LegalAgentState) -> dict:
    """
    Handles immediate-deadline cases:
    - Prepares attorney alerts
    - Drafts an urgent triage message for the client
    """
    intake = state["intake_data"]
    client_name = intake.client_name or "Client"

    # In production, trigger Slack/Twilio/Email webhooks to the duty lawyer here
    print(f"\n[ALERT DISPATCHED]: Emergency alert sent to lead counsel for {client_name}!")

    urgent_draft = (
        f"Hello {client_name}, we have flagged your inquiry as URGENT "
        f"due to an imminent deadline/hearing concerning your {intake.legal_domain} matter. "
        "An on-duty attorney has been directly notified and will contact you shortly. "
        "Please preserve all documentation, summons, and messages."
    )

    return {
        "draft_response": urgent_draft,
        "is_escalated": True
    }


def standard_flow_node(state: LegalAgentState) -> dict:
    """
    Placeholder for the standard path:
    Where the Hybrid RAG statutory lookup runs for non-urgent matters.
    """
    intake = state["intake_data"]
    client_name = intake.client_name or "Client"

    standard_draft = (
        f"Hello {client_name}, thank you for contacting us regarding your {intake.legal_domain} issue. "
        "Your request is being reviewed under our standard intake workflow. "
        f"Search keywords prepared for statutory lookup: '{intake.search_keywords}'."
    )

    return {
        "draft_response": standard_draft,
        "is_escalated": False
    }


# ==========================================
# 2. DEFINE THE CONDITIONAL ROUTER FUNCTION
# ==========================================

def urgency_router(state: LegalAgentState) -> str:
    """
    Evaluates the state after intake_node runs.
    Returns the exact name of the destination node.
    """
    if state.get("is_escalated"):
        return "escalate_urgent"
    return "standard_flow"




In [17]:
# ==========================================
# 3. ASSEMBLE AND COMPILE THE STATEGRAPH
# ==========================================

builder = StateGraph(LegalAgentState)

# Register nodes
builder.add_node("intake", intake_node)
builder.add_node("escalate_urgent", escalate_urgent_node)
builder.add_node("standard_flow", standard_flow_node)

# Connect edges
builder.add_edge(START, "intake")

# Branch conditionally out of the intake node
builder.add_conditional_edges(
    "intake",
    urgency_router,
    {
        "escalate_urgent": "escalate_urgent",
        "standard_flow": "standard_flow"
    }
)

# Connect leaf nodes to the END marker
builder.add_edge("escalate_urgent", END)
builder.add_edge("standard_flow", END)

# Compile into an executable graph application
app = builder.compile()

In [18]:
urgent_input = {
    "messages": [{
        "role": "user",
        "content": "My landlord kicked me out without an order and I have a police summons tomorrow morning at 8 AM!"
    }],
    "sender_id": "+201000000001",
    "intake_data": None,
    "retrieved_statutes": [],
    "draft_response": "",
    "is_escalated": False
}

urgent_result = app.invoke(urgent_input)

print("\n--- CASE 1 (CRITICAL) RESULT ---")
print("Target Node Hit :", "escalate_urgent" if urgent_result["is_escalated"] else "standard_flow")
print("Is Escalated    :", urgent_result["is_escalated"])
print("WhatsApp Draft  :\n", urgent_result["draft_response"])


[ALERT DISPATCHED]: Emergency alert sent to lead counsel for Client!

--- CASE 1 (CRITICAL) RESULT ---
Target Node Hit : escalate_urgent
Is Escalated    : True
WhatsApp Draft  :
 Hello Client, we have flagged your inquiry as URGENT due to an imminent deadline/hearing concerning your Civil matter. An on-duty attorney has been directly notified and will contact you shortly. Please preserve all documentation, summons, and messages.


In [19]:
standard_input = {
    "messages": [{
        "role": "user",
        "content": "Hi, I want to review an employment contract clause regarding 30 days notice period before signing next month."
    }],
    "sender_id": "+201000000002",
    "intake_data": None,
    "retrieved_statutes": [],
    "draft_response": "",
    "is_escalated": False
}

standard_result = app.invoke(standard_input)

print("\n--- CASE 2 (STANDARD) RESULT ---")
print("Target Node Hit :", "escalate_urgent" if standard_result["is_escalated"] else "standard_flow")
print("Is Escalated    :", standard_result["is_escalated"])
print("WhatsApp Draft  :\n", standard_result["draft_response"])


--- CASE 2 (STANDARD) RESULT ---
Target Node Hit : standard_flow
Is Escalated    : False
WhatsApp Draft  :
 Hello Client, thank you for contacting us regarding your Labor issue. Your request is being reviewed under our standard intake workflow. Search keywords prepared for statutory lookup: 'employment contract review, 30 days notice period, labor law, contract clause'.


#Hybrid RAG Retriever.

In [20]:
# ==========================================
# 4. DEFINE THE RAG RETRIEVAL NODE (MOCK)
# ==========================================

def retrieve_statutes_node(state: LegalAgentState) -> dict:
    """
    Simulates a PostgreSQL Hybrid RAG search (pgvector + tsvector via RRF).
    In production, this executes the SQL query against your DB.
    """
    intake = state["intake_data"]
    query = intake.search_keywords.lower()
    domain = intake.legal_domain

    print(f"\n[RAG SEARCH TRIGGERED] Executing Hybrid Search for: '{query}' in {domain} law...")

    # Mock Database: A small dictionary representing statutory knowledge
    mock_database = {
        "employment notice period 30 days contract": [
            "Labor Law Art. 104: Either party may terminate an employment contract by providing 30 days written notice.",
            "Labor Law Art. 105: If notice is not provided, compensation equal to the wage for the notice period is due."
        ],
        "eviction lock out police summons": [
            "Civil Code Art. 560: A landlord may not evict a tenant or change locks without a formal court eviction order.",
            "Criminal Code Art. 320: Unauthorized self-help evictions involving property seizure may result in police intervention."
        ]
    }

    # Basic matching logic to simulate search hits
    retrieved_results = []
    for key, laws in mock_database.items():
        # If any significant keyword from the query matches the database key
        if any(word in key for word in query.split() if len(word) > 3):
            retrieved_results.extend(laws)

    # Fallback if no specific laws are found
    if not retrieved_results:
        retrieved_results = [f"General {domain} principle: Refer to primary statutory code for specific provisions regarding '{query}'."]

    return {
        "retrieved_statutes": retrieved_results
    }




In [21]:
# ==========================================
# 5. DEFINE THE RESPONSE SYNTHESIS NODE
# ==========================================

def draft_response_node(state: LegalAgentState) -> dict:
    """
    Uses the LLM to draft the final WhatsApp message,
    strictly grounded in the retrieved statutes.
    """
    intake = state["intake_data"]
    statutes = "\n".join(state["retrieved_statutes"])
    client_name = intake.client_name or "Client"

    prompt = f"""
    You are a legal intake assistant. Draft a professional WhatsApp response to the client.

    Client Name: {client_name}
    Legal Issue: {intake.case_summary}

    Retrieved Statutory Context:
    {statutes}

    Instructions:
    1. Acknowledge the client's issue.
    2. Provide the relevant legal information based STRICTLY on the Retrieved Statutory Context.
    3. Do NOT promise a specific legal outcome.
    4. End with this exact disclaimer: "*Disclaimer: This is for informational purposes only and does not constitute formal legal advice.*"
    """

    # Call the LLM to generate the final text
    response = llm.invoke(prompt)

    return {
        "draft_response": response.content
    }


In [22]:

# ==========================================
# 6. RE-COMPILE THE STATEGRAPH
# ==========================================

# Create a fresh builder
builder = StateGraph(LegalAgentState)

# Add all nodes
builder.add_node("intake", intake_node)
builder.add_node("escalate_urgent", escalate_urgent_node)
builder.add_node("retrieve_statutes", retrieve_statutes_node)  # New Node
builder.add_node("draft_response", draft_response_node)          # New Node

# Define the flow
builder.add_edge(START, "intake")

# The router decides: Escalate OR Retrieve
builder.add_conditional_edges(
    "intake",
    urgency_router,
    {
        "escalate_urgent": "escalate_urgent",
        "standard_flow": "retrieve_statutes" # Map standard flow to the RAG node
    }
)

# After retrieving statutes, draft the response
builder.add_edge("retrieve_statutes", "draft_response")

# Both paths end the workflow
builder.add_edge("escalate_urgent", END)
builder.add_edge("draft_response", END)

app = builder.compile()

In [23]:
standard_input = {
    "messages": [{
        "role": "user",
        "content": "Hi, I want to review an employment contract clause regarding 30 days notice period before signing next month."
    }],
    "sender_id": "+201000000002",
    "intake_data": None,
    "retrieved_statutes": [],
    "draft_response": "",
    "is_escalated": False
}

result = app.invoke(standard_input)
print("\n--- FINAL WHATSAPP MESSAGE ---")
print(result["draft_response"])


[RAG SEARCH TRIGGERED] Executing Hybrid Search for: 'employment contract review, 30 days notice period, labor law, contract clause' in Labor law...

--- FINAL WHATSAPP MESSAGE ---
Hi Client,

Thank you for reaching out. I understand you would like to review the 30-day notice period clause in your employment contract before signing next month.

Regarding notice periods, Labor Law Art. 104 states that either party may terminate an employment contract by providing 30 days written notice. Furthermore, Labor Law Art. 105 indicates that if notice is not provided, compensation equal to the wage for the notice period is due.

We can certainly assist you with reviewing your contract. Please let us know when you are ready to proceed.

*Disclaimer: This is for informational purposes only and does not constitute formal legal advice.*
